# 018번 감성대화말뭉치 데이터셋 탐색

**목적**: `emotional_peak` 경계 케이스 발화 예시 추출  
**활용처**: `tag-utterances` few-shot — 감정이 폭발적으로 드러나는 발화 vs 일반 감정 발화 구분 기준  
**Drive 경로**: `내드라이브/Dadam_dataSet/018.감성대화/Validation/라벨링데이터/감성대화말뭉치(최종데이터)_Validation.zip`  

## 처리 흐름
```
Cell 1 → Drive 마운트
Cell 2 → 경로 설정 및 zip 파일 확인
Cell 3 → JSON 구조 파악
Cell 4 → 전체 데이터 로드 (감정 레이블 포함)
Cell 5 → emotional_peak 해당 발화 필터링
Cell 6 → 감정별 경계 케이스 예시 선별
Cell 7 → few-shot JSON 저장
```

## Cell 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2. 경로 설정 및 zip 확인

In [ ]:
import os
import zipfile

# Drive 경로 직접 지정 (스크린샷 기준)
BASE_PATH = '/content/drive/MyDrive/Dadam_dataSet/018.감성대화/Validation/라벨링데이터'
ZIP_NAME = '감성대화말뭉치(최종데이터)_Validation.zip'
ZIP_PATH = os.path.join(BASE_PATH, ZIP_NAME)

print('경로 존재 여부:', os.path.exists(ZIP_PATH))

# 경로가 없을 경우 자동 탐색
if not os.path.exists(ZIP_PATH):
    MYDRIVE = '/content/drive/MyDrive'
    print('\n[자동 탐색] MyDrive에서 018 폴더 검색 중...')
    for dirpath, dirnames, filenames in os.walk(MYDRIVE):
        depth = dirpath.replace(MYDRIVE, '').count(os.sep)
        if depth > 5:
            del dirnames[:]
            continue
        for fname in filenames:
            if '감성대화말뭉치' in fname and fname.endswith('.zip'):
                ZIP_PATH = os.path.join(dirpath, fname)
                print(f'발견: {ZIP_PATH}')

# zip 내부 파일 목록 미리보기
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    all_files = z.namelist()

print(f'\nzip 내 파일 총 {len(all_files)}개')
print('--- 처음 10개 ---')
for f in all_files[:10]:
    print(f)

## Cell 3. JSON 구조 파악

018번은 046번과 달리 단일 JSON 파일에 모든 대화가 포함되어 있을 가능성이 높음.  
감정 레이블, 감정 강도(0~3 or 1~3) 필드 위치 확인이 핵심.

In [ ]:
import json

# zip에서 JSON 파일 하나 열어 최상위 구조 확인
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    json_files = [f for f in z.namelist() if f.endswith('.json')]
    print(f'JSON 파일 수: {len(json_files)}개')

    with z.open(json_files[0]) as f:
        raw = json.load(f)

# 최상위 타입 확인 (list 또는 dict)
print('최상위 타입:', type(raw).__name__)

if isinstance(raw, list):
    print(f'리스트 항목 수: {len(raw)}개')
    first_item = raw[0]
    print('첫 번째 항목 타입:', type(first_item).__name__)
    if isinstance(first_item, dict):
        print('첫 번째 항목 키:', list(first_item.keys()))
        # 중첩 구조 한 단계 더 확인
        for key, val in first_item.items():
            if isinstance(val, dict):
                print(f'  [{key}] 하위 키:', list(val.keys()))
            elif isinstance(val, list):
                print(f'  [{key}] 리스트 {len(val)}개')
            else:
                print(f'  [{key}]:', val)
elif isinstance(raw, dict):
    print('최상위 키:', list(raw.keys()))
    for key in raw.keys():
        val = raw[key]
        if isinstance(val, list):
            print(f'\n배열 키 [{key}]: {len(val)}개 항목')
            if len(val) > 0:
                print('첫 번째 항목 키:', list(val[0].keys()) if isinstance(val[0], dict) else type(val[0]))
        elif isinstance(val, dict):
            print(f'\n딕셔너리 키 [{key}]:', list(val.keys())[:10])

## Cell 4. 감정 대화 구조 상세 확인

018번 감성대화 데이터셋은 다음 구조를 가질 것으로 예상:
- `profile`: 발화자 정보 (나이대, 성별 등)
- `talk.content.HS01~HS10`: 사람 발화 (Human Speaker)
- `talk.content.SS01~SS10`: 시스템 응답 (System Speaker)
- `profile.emotion.type`: 감정 유형
- `profile.emotion.level`: 감정 강도

→ 실제 구조는 Cell 3 출력 결과로 확인 후 아래 파싱 코드 수정

In [ ]:
# 018번 감정 코드 매핑 확정
# 코드 번호 대분류(E1x=기쁨 등)가 실제 발화와 불일치 확인
# → 코드 기반 분류 불가, 발화 텍스트 키워드로 직접 감정 분류

# emotion-id 에서 'type' 대신 situation 코드도 확인
sample = raw[0]
print('emotion 필드 전체:', sample['profile']['emotion'])
print()

# 전체 emotion type 코드별 발화 1개씩 → 그룹 패턴 재확인
from collections import defaultdict
code_samples = defaultdict(list)
for item in raw:
    code = item['profile']['emotion'].get('type', '')
    hs01 = item['talk'].get('content', {}).get('HS01', '')
    if hs01:
        code_samples[code].append(hs01)

# E6x(기쁨)와 나머지 샘플 10개 비교
print('=== E6x (기쁨으로 확인된 그룹) ===')
for code in ['E60', 'E61', 'E62']:
    print(f'[{code}] {code_samples[code][0][:70]}')

print('\n=== E1x~E5x (부정 감정 그룹 — 세부 분류 불명확) ===')
for code in ['E10', 'E20', 'E30', 'E40', 'E50']:
    print(f'[{code}] {code_samples[code][0][:70]}')

## Cell 5. 전체 데이터 로드 및 파싱

018번 감성대화말뭉치 표준 구조 (AI Hub 공식 문서 기준):
```json
{
  "profile": {
    "persona-id": "...",
    "emotion": {
      "type": "기쁨",        // 감정 유형
      "level": 1            // 강도: 1=보통, 2=강함, 3=매우강함 (또는 0~3)
    },
    "age": "노년"
  },
  "talk": {
    "id": "...",
    "content": {
      "HS01": "사람 발화 1",
      "SS01": "시스템 응답 1",
      ...
    }
  }
}
```

**emotional_peak 기준**: 감정 강도 최고값(level=3 or level=2+특정 감정)  
→ 발화 내용에 감탄사·눈물 언급·절망 표현이 수반되는 것

In [ ]:
import pandas as pd
import re

# 발화 텍스트 기반 감정 키워드 분류
# 목적: emotional_peak 경계 케이스 추출 → 명확히 감정 드러나는 발화만 신뢰
# 기타(키워드 미매칭) 비율이 높아도 OK — 명확한 발화만 활용

EMOTION_KEYWORDS = {
    '기쁨': [
        r'기쁘', r'기뻐', r'행복해', r'행복했', r'뿌듯', r'즐거워', r'즐거웠',
        r'신나', r'신났', r'반가워', r'다행이', r'안도했', r'흐뭇', r'보람',
        r'자랑스러', r'유쾌', r'설레', r'설렜',
    ],
    '슬픔': [
        r'슬퍼', r'슬펐', r'슬프다', r'우울해', r'우울했', r'눈물', r'울었',
        r'외로워', r'외로웠', r'그리워', r'허전', r'씁쓸', r'서글', r'속상해',
        r'속상했', r'안타까워', r'무기력', r'절망', r'포기하고',
    ],
    '분노': [
        r'화가 나', r'화가 났', r'화났어', r'짜증 나', r'짜증났', r'열받',
        r'부당해', r'분해', r'억울해', r'억울했', r'불만이', r'불쾌',
        r'어이없', r'황당해', r'빡쳐', r'열 받',
    ],
    '불안': [
        r'불안해', r'불안했', r'걱정돼', r'걱정됐', r'무서워', r'무서웠',
        r'두렵', r'두려워', r'두려웠', r'떨려', r'겁이 나', r'겁났',
        r'조마조마', r'초조해', r'염려',
    ],
    '당황': [
        r'당황했', r'당황스러', r'창피해', r'창피했', r'부끄러워', r'부끄러웠',
        r'민망해', r'민망했', r'충격이었', r'충격을 받', r'놀랐어', r'놀라서',
        r'어리둥절', r'난감해',
    ],
    '상처': [
        r'상처받', r'실망했', r'실망이야', r'서운해', r'서운했', r'배신당',
        r'후회해', r'후회됐', r'자책', r'죄책감', r'괴로워', r'괴로웠',
        r'마음이 아파', r'가슴이 아파', r'낙담', r'좌절',
    ],
}

def classify_emotion(text):
    """발화 텍스트에서 감정 키워드 기반 분류 — 명확한 감정 표현만 매칭"""
    for emotion in ['분노', '슬픔', '불안', '상처', '당황', '기쁨']:
        for p in EMOTION_KEYWORDS[emotion]:
            if re.search(p, text):
                return emotion
    return '기타'

def parse_018_data(raw_list):
    """018번 감성대화말뭉치 JSON 리스트 → DataFrame 변환"""
    rows = []
    for item in raw_list:
        talk = item.get('talk', {})
        content = talk.get('content', {})
        emotion_code = item['profile']['emotion'].get('type', '')

        for i in range(1, 10):
            hs_key = f'HS0{i}'
            ss_key = f'SS0{i}'
            human_text = content.get(hs_key, '')
            system_text = content.get(ss_key, '')
            if not human_text:
                continue

            emotion_type = classify_emotion(human_text)
            rows.append({
                'emotion_code': emotion_code,
                'emotion_type': emotion_type,
                'turn': i,
                'human': human_text,
                'system': system_text,
                'dialog_id': talk.get('id', {}).get('talk-id', ''),
            })

    return pd.DataFrame(rows)


df = parse_018_data(raw)

# 기타 제외 후 실사용 데이터
df_valid = df[df['emotion_type'] != '기타'].copy()

print(f'전체 발화: {len(df)}개')
print(f'감정 명시 발화 (기타 제외): {len(df_valid)}개 ({len(df_valid)/len(df)*100:.1f}%)')
print('\n[감정별 분포]')
print(df_valid['emotion_type'].value_counts())

print('\n[미리보기 — 감정별 2개]')
for emotion in ['기쁨', '슬픔', '분노', '불안', '당황', '상처']:
    rows = df_valid[df_valid['emotion_type'] == emotion]
    if len(rows) == 0:
        continue
    print(f'\n[{emotion}]')
    for _, r in rows.head(2).iterrows():
        print(f'  {r["human"]}')

## Cell 6. emotional_peak 발화 필터링

**emotional_peak 판단 기준** (tag-utterances 프롬프트 개선용):
1. `emotion_level` 최고값 (3 또는 최대값)
2. 발화 내 감정 강도 키워드 포함:
   - 감탄사: `"어머", "세상에", "아이고", "맙소사"`
   - 눈물·울음: `"눈물", "울었", "울고"`
   - 절망·고통: `"못 살겠", "힘들어 죽겠", "너무너무"`
   - 기쁨 최고조: `"얼마나 기쁜지", "정말 행복", "너무 좋아"`
3. 발화 길이 20자 이상

**non-peak 감정 발화** (경계 케이스 비교용):
- 같은 감정이지만 level이 낮거나 평서문 형태

In [ ]:
# emotional_peak 발화 필터링
# 018번은 emotion_level 없음 → 키워드 기반으로만 peak 판단

PEAK_KEYWORDS = [
    # 감탄사
    r'어머', r'세상에', r'아이고', r'맙소사', r'에구', r'에고',
    # 눈물·울음
    r'눈물', r'울었', r'울고', r'눈물이', r'눈물 나',
    # 강도 부사
    r'너무너무', r'정말정말', r'얼마나.*지',
    # 신체 반응
    r'가슴이', r'심장이', r'온몸이',
    # 절망·고통
    r'못 살겠', r'힘들어 죽겠', r'살기 싫', r'죽고 싶',
    # 기쁨 최고조
    r'얼마나 기쁜', r'정말 행복', r'꿈인지', r'너무 좋아서',
    # 분노 최고조
    r'참을 수가 없', r'도저히',
    # 슬픔 최고조
    r'너무 슬퍼', r'너무 외로', r'너무 힘들어', r'너무 속상',
    # 불안 최고조
    r'너무 무서워', r'너무 불안', r'너무 두려',
    # 당황 최고조
    r'너무 당황', r'너무 창피', r'너무 충격',
    # 상처 최고조
    r'너무 실망', r'너무 서운', r'너무 상처',
]

peak_pattern = '|'.join(PEAK_KEYWORDS)

# df_valid 기준으로 peak / non-peak 분리
df_valid['has_peak_keyword'] = df_valid['human'].str.contains(peak_pattern, na=False)

df_peak = df_valid[
    df_valid['has_peak_keyword'] &
    (df_valid['human'].str.len() >= 20)
].copy()

df_nonpeak = df_valid[
    ~df_valid['has_peak_keyword'] &
    (df_valid['human'].str.len() >= 20)
].copy()

print(f'emotional_peak 해당 발화: {len(df_peak)}개')
print(df_peak['emotion_type'].value_counts())

print(f'\nnon-peak 발화 (경계 비교용): {len(df_nonpeak)}개')
print(df_nonpeak['emotion_type'].value_counts())

print('\n=== emotional_peak 발화 샘플 (감정별 1개) ===')
for emotion in ['기쁨', '슬픔', '분노', '불안', '당황', '상처']:
    rows = df_peak[df_peak['emotion_type'] == emotion]
    if len(rows) == 0:
        print(f'\n[{emotion}] peak 없음')
        continue
    r = rows.iloc[0]
    print(f'\n[{emotion}]')
    print(f'  발화: {r["human"]}')

## Cell 7. 경계 케이스 쌍 구성 (peak vs non-peak)

tag-utterances few-shot에는 두 가지 예시가 필요:
1. **peak 발화** → `["emotional_peak"]` 태그 부여
2. **같은 감정이지만 non-peak** → `["daily_mundane"]` 또는 `["memory_recall"]` 태그 (emotional_peak 없음)

이 대비가 경계 케이스 학습에 핵심.

In [ ]:
# 감정별로 peak 1개 + non-peak 1개를 쌍으로 구성
TARGET_EMOTIONS = ['기쁨', '슬픔', '분노', '불안', '당황', '상처']

boundary_cases = []

for emotion in TARGET_EMOTIONS:
    peak_pool = df_peak[
        (df_peak['emotion_type'] == emotion) &
        (df_peak['human'].str.len() >= 25) &
        (df_peak['human'].str.len() <= 120)
    ]

    nonpeak_pool = df_nonpeak[
        (df_nonpeak['emotion_type'] == emotion) &
        (df_nonpeak['human'].str.len() >= 25) &
        (df_nonpeak['human'].str.len() <= 100)
    ]

    if len(peak_pool) == 0:
        print(f'[{emotion}] peak 발화 없음 — 건너뜀')
        continue
    if len(nonpeak_pool) == 0:
        print(f'[{emotion}] non-peak 발화 없음 — peak만 포함')

    peak_row = peak_pool.sort_values(
        'human', key=lambda x: x.str.len(), ascending=False
    ).iloc[0]

    boundary_cases.append({
        'emotion': emotion,
        'is_peak': True,
        'utterance': peak_row['human'],
        'has_peak_keyword': peak_row['has_peak_keyword'],
        'suggested_tags': ['emotional_peak'],
    })

    if len(nonpeak_pool) > 0:
        nonpeak_row = nonpeak_pool.iloc[0]
        boundary_cases.append({
            'emotion': emotion,
            'is_peak': False,
            'utterance': nonpeak_row['human'],
            'has_peak_keyword': False,
            'suggested_tags': ['daily_mundane'],  # TASK-04에서 수동 검토 후 수정
        })

print(f'경계 케이스 발화 총 {len(boundary_cases)}개 구성')

for case in boundary_cases:
    peak_label = '🔴 emotional_peak' if case['is_peak'] else '⚪ non-peak'
    print(f'\n[{case["emotion"]}] {peak_label}')
    print(f'발화: {case["utterance"]}')
    print(f'제안 태그: {case["suggested_tags"]}')

## Cell 8. few-shot JSON 저장

저장 형식 (tag-utterances few-shot용):
```json
[
  {
    "emotion": "기쁨",
    "is_emotional_peak": true,
    "utterance": "...",
    "tags": ["emotional_peak"]
  },
  ...
]
```

**TASK-04 작업 시 활용**:
- `is_emotional_peak: true` 항목 → emotional_peak 태그 부여 근거 예시
- `is_emotional_peak: false` 항목 → emotional_peak 제외 기준 예시

In [ ]:
import json

# few-shot JSON 형식으로 변환
fewshot_018 = []
for case in boundary_cases:
    fewshot_018.append({
        'emotion': case['emotion'],
        'is_emotional_peak': case['is_peak'],
        'utterance': case['utterance'],
        'tags': case['suggested_tags'],
        'note': f'keyword={case["has_peak_keyword"]}',
    })

# Drive 저장
OUTPUT_DRIVE = '/content/drive/MyDrive/Dadam_dataSet/018_fewshot_boundary.json'
with open(OUTPUT_DRIVE, 'w', encoding='utf-8') as f:
    json.dump(fewshot_018, f, ensure_ascii=False, indent=2)

print(f'Drive 저장 완료: {OUTPUT_DRIVE}')
print(f'총 {len(fewshot_018)}개 경계 케이스 발화')
print('\n[저장된 내용 미리보기]')
for item in fewshot_018:
    peak_label = '🔴 peak' if item['is_emotional_peak'] else '⚪ non-peak'
    print(f"[{item['emotion']}] {peak_label}: {item['utterance'][:60]}...")

## Cell 9. 로컬 다운로드

Drive 저장 완료 후 로컬로도 다운로드해서 `prompt/fewshot/` 에 저장

In [ ]:
from google.colab import files

# /content에 로컬 복사본 저장
LOCAL_PATH = '/content/018_fewshot_boundary.json'
with open(LOCAL_PATH, 'w', encoding='utf-8') as f:
    json.dump(fewshot_018, f, ensure_ascii=False, indent=2)

# 다운로드 (prompt/fewshot/018_fewshot_boundary.json 으로 저장)
files.download(LOCAL_PATH)
print('다운로드 완료 → prompt/fewshot/018_fewshot_boundary.json 에 저장')